### Import libraries

In [10]:
from pathlib import Path
import pandas as pd

### Set file paths

In [11]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "datasets" / "raw"
OUT = ROOT / "datasets" / "processed" / "Combaine_paddy.csv"

###  Load the two datasets

In [12]:
pair_file = RAW / "AgriMind_Bilingual_Paddy_Farmer_QA_12720_Pairs.csv"
rows_file = RAW / "AgriMind_Sinhala_English_Paddy_Farmer_QA_25440.csv"
assert pair_file.exists() and rows_file.exists(), "Place both source CSV files in datasets/raw/."
pairs = pd.read_csv(pair_file, keep_default_na=False, encoding="utf-8")
rows = pd.read_csv(rows_file, keep_default_na=False, encoding="utf-8")

print("Bilingual pairs:", pairs.shape)
print("Language-specific rows:", rows.shape)

Bilingual pairs: (12720, 15)
Language-specific rows: (25440, 16)


### Separate Sinhala and English data

In [13]:
metadata = ["pair_id", "category_en", "category_si", "issue_id", "issue_en", "crop_stage_en", "crop_stage_si", "urgency", "source_url", "data_origin", "review_status"]

si = pairs[metadata + ["question_si", "answer_si"]].rename(columns={"question_si":"question", "answer_si":"answer"}).copy()
si["language"] = "si-LK"
si["qa_id"] = si["pair_id"].str.replace("PADDY_PAIR_", "PADDY_SI_", regex=False)
si["is_real_farmer_query"] = False

en = pairs[metadata + ["question_en", "answer_en"]].rename(columns={"question_en":"question", "answer_en":"answer"}).copy()
en["language"] = "en-LK"
en["qa_id"] = en["pair_id"].str.replace("PADDY_PAIR_", "PADDY_EN_", regex=False)
en["is_real_farmer_query"] = False

long_from_pairs = pd.concat([si, en], ignore_index=True)

###  Compare source representations

In [14]:
comparison_cols = [
    "question",
    "answer",
    "category_en",
    "category_si",
    "issue_id",
    "issue_en",
    "crop_stage_en",
    "crop_stage_si",
    "urgency",
    "source_url",
    "data_origin",
    "review_status"
]

left = long_from_pairs[["pair_id", "language"] + comparison_cols].copy()
right = rows[["pair_id", "language"] + comparison_cols].copy()

for frame in [left, right]:
    for col in frame.select_dtypes(include="object").columns:
        frame[col] = frame[col].astype(str).str.strip()

check = left.merge(
    right,
    on=["pair_id", "language"],
    how="outer",
    suffixes=("_pair", "_rows"),
    indicator=True,
    validate="one_to_one"
)

assert (check["_merge"] == "both").all(), (
    "Some pair/language keys do not match. Inspect the source CSV files."
)

for col in comparison_cols:
    assert (check[f"{col}_pair"] == check[f"{col}_rows"]).all(), (
        f"Mismatch detected in {col}."
    )


combined = rows.copy()

for col in combined.select_dtypes(include="object").columns:
    combined[col] = combined[col].astype(str).str.strip()

combined["is_real_farmer_query"] = (
    combined["is_real_farmer_query"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

combined = combined[
    [
        "qa_id",
        "pair_id",
        "question",
        "answer",
        "language",
        "category_en",
        "category_si",
        "issue_id",
        "issue_en",
        "crop_stage_en",
        "crop_stage_si",
        "urgency",
        "source_url",
        "data_origin",
        "review_status",
        "is_real_farmer_query"
    ]
]

assert not combined.duplicated(["pair_id", "language"]).any()

assert combined["pair_id"].nunique() == pairs["pair_id"].nunique()

assert combined["language"].value_counts().to_dict() == {
    "si-LK": 12720,
    "en-LK": 12720
}

OUT.parent.mkdir(parents=True, exist_ok=True)

combined.to_csv(
    OUT,
    index=False,
    encoding="utf-8-sig"
)

print("Final shape:", combined.shape)


C:\Users\USER\AppData\Local\Temp\ipykernel_18776\3962114278.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in frame.select_dtypes(include="object").columns:
C:\Users\USER\AppData\Local\Temp\ipykernel_18776\3962114278.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gui

Final shape: (25440, 16)
